# 1. What Dependency Injection Is in FastAPI

Dependency Injection (DI) is a design pattern where a function or class receives its dependencies from external sources rather than creating them internally.

**In FastAPI:**

- DI helps you separate concerns, keep routes thin, and reuse logic.

- Dependencies are automatically resolved and injected by FastAPI.

- Common use cases: database sessions, authentication, configuration, validation, etc.

# 2. Defining and Using Depends()

Depends is the core tool for DI in FastAPI.

In [ ]:
from fastapi import FastAPI, Depends

app = FastAPI()

# A simple dependency function
def get_query_param(q: str = "default"):
    return q

# Using dependency in a route
@app.get("/items/")
def read_items(query: str = Depends(get_query_param)):
    return {"query": query}

**Key points:**

- Depends() tells FastAPI to call the function and inject its return value.
- Can be used in route parameters, other dependencies, or classes.

# 3. Using Dependencies for Database Sessions, Config, and Shared Logic

## **Example: Database Session Dependency**

In [ ]:
from sqlalchemy.orm import Session
from database import SessionLocal

# Dependency to provide DB session
def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()

@app.get("/users/")
def get_users(db: Session = Depends(get_db)):
    return db.query(User).all()

- yield is used for cleanup (closing DB connection).

- FastAPI ensures one DB session per request.

## **Example: Configuration Dependency**

In [ ]:
def get_settings():
    return {"api_key": "mysecret"}

@app.get("/config")
def read_config(settings = Depends(get_settings)):
    return settings

# 4. Using Dependencies for Reusable Validation and Authentication
## Validation Example

In [ ]:
from fastapi import HTTPException

def validate_positive(number: int):
    if number <= 0:
        raise HTTPException(status_code=400, detail="Must be positive")
    return number

@app.get("/square/")
def get_square(n: int = Depends(validate_positive)):
    return {"square": n**2}

## Authentication Example

In [ ]:
from fastapi import Security

def get_current_user(token: str):
    # Validate token here
    if token != "secret":
        raise HTTPException(status_code=401, detail="Unauthorized")
    return {"username": "john_doe"}

@app.get("/me/")
def read_me(user: dict = Depends(get_current_user)):
    return user

# 5. Dependency Scope and Order (Conceptual)

- Dependencies can call other dependencies, creating a tree of injections.

- Scope determines how often the dependency is run:

    - per-request (default)

    - singleton (shared across requests, using caching)

- Execution order:

    1. Dependencies are resolved recursively from the leaves up.

    2. The final return values are injected into the route parameters.

**Example of nested dependencies:**

In [ ]:
def dep_a():
    return "A"

def dep_b(a: str = Depends(dep_a)):
    return a + "B"

@app.get("/nested")
def nested_route(b: str = Depends(dep_b)):
    return b  # Returns "AB"